In [ ]:
import kagglehub
import os
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
import seaborn as sns
from PIL import Image
from tqdm import tqdm

In [ ]:
# Get the dataset path
path = kagglehub.dataset_download("gpiosenka/sports-classification")

print(f"Dataset location: {path}\n")

### Core Statistics

(Add file size checker to see the sizes for all images, the say smt like "imgaes range from xxx to xxx in terms of size")

In [ ]:
# Load and analyze CSV
df = pd.read_csv(os.path.join(path, "sports.csv"))
print(df.head())
print("\n" + "="*60)

# Analyze folder structure and file types from actual filesystem
print(f"\nContents of directory:")
print(os.listdir(path))

print("\n" + "="*60)
print("FOLDER STRUCTURE:\n")

folder_counts = defaultdict(int)
file_type_counts = defaultdict(int)

for root, dirs, files in os.walk(path):
    for file in files:
        # Get the relative path from path
        rel_path = os.path.relpath(os.path.join(root, file), path)
        top_folder = rel_path.split(os.sep)[0]
        folder_counts[top_folder] += 1
        
        # Get file extension
        _, ext = os.path.splitext(file)
        if ext:  # Only count if there's an extension
            file_type_counts[ext] += 1

for folder in sorted(folder_counts.keys()):
    print(f"  {folder}/: {folder_counts[folder]} files")

print("\nFILE TYPES:")
for ext in sorted(file_type_counts.keys()):
    print(f"  {ext}: {file_type_counts[ext]} files")

# Basic statistics
print(f"TOTAL IMAGES: {len(df)}")
print(f"TOTAL CLASSES: {df['labels'].nunique()}")

# Visualize class distribution
counts = df["labels"].value_counts()
plt.figure(figsize=(10, 15))
sns.barplot(y=counts.index, x=counts.values, hue=counts.index, palette='inferno', legend=False)
plt.title('Amount of images per label')
plt.ylabel('Labels')
plt.xlabel('Count')
for i, v in enumerate(counts.values):
    plt.text(v + 0.5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig("dataset_distribution_full.png", dpi=150)
plt.show()

### Data Quality & Characteristics

In [ ]:
# Calculate image sizes
print("\nCalculating image sizes...")
 
def get_image_size(filepath):
    try:
        with Image.open(filepath) as img:
            width, height = img.size
            return [width, height]  # Store as [width, height]
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None
 
df["image_size"] = [
    get_image_size(os.path.join(path, fp)) 
    for fp in tqdm(df["filepaths"], desc="Files")
]
 
# Display statistics
print(f"\nImage sizes added!")
 
# Extract widths and heights for statistics
widths = df["image_size"].apply(lambda x: x[0] if x else None)
heights = df["image_size"].apply(lambda x: x[1] if x else None)
pixel_counts = df["image_size"].apply(lambda x: x[0] * x[1] if x else None)
 
print(f"Width  - Min: {widths.min()}, Max: {widths.max()}, Mean: {widths.mean():.0f}")
print(f"Height - Min: {heights.min()}, Max: {heights.max()}, Mean: {heights.mean():.0f}")
print(f"Resolution - Min: {pixel_counts.min()}, Max: {pixel_counts.max()}, Mean: {pixel_counts.mean():.0f}")
print(f"Missing sizes: {df['image_size'].isna().sum()} images")

### Class-Specific Insights

In [ ]:
# Per-class statistics:
    # DONE Number of images per class
    # UNNECESARY Mean/min/max image dimensions per class
    # DONE Class imbalance ratio (largest class / smallest class)
# Identify minority/majority classes

# Class imbalance
# class_imbalance = df.groupby('labels').size().reset_index(name='count').max() / df.groupby('labels').size().reset_index(name='count').min()
grouped = df.groupby('labels').size().reset_index(name='count')
max_idx = grouped['count'].idxmax()
min_idx = grouped['count'].idxmin()

max_row = grouped.loc[max_idx]
name_max = max_row['labels']
count_max = max_row['count']

min_row = grouped.loc[min_idx]
name_min = min_row['labels']
count_min = min_row['count']

print(f"The {name_max} class ({count_max} images) is {count_max/count_min:.2f} times bigger than the {name_min} class ({count_min} images).")

### The code that's below here, is stuff that I'm working on atm, it doesn't have a "category"

In [ ]:
# Load and display an image
img_path = os.path.join(path, "test/air hockey/1.jpg")
img = imread(img_path)

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title("Air Hockey")
plt.show()

In [ ]:
# Get summary stats
summary = df.groupby('labels').size().reset_index(name='count')
summary = summary.sort_values('count', ascending=False)
print(summary[0:10])

### The same but for specific classes

In [ ]:
def visualize_sports_distribution(df, sports_list, figsize=(8, 4), palette='inferno', save_path="dataset_distribution.png"):
    """
    Visualize class distribution for selected sports.
    
    Parameters:
    - df: DataFrame with 'labels' column
    - sports_list: List of sports to filter
    - figsize: Figure size (default: (8, 4))
    - palette: Color palette (default: 'inferno')
    - save_path: Path to save the figure (default: "dataset_distribution.png")
    """
    
    # Filter dataframe
    filtered_df = df[df['labels'].isin(sports_list)]
    
    print(filtered_df["labels"].unique())
    print(f"\nOriginal: {len(df)} rows")
    print(f"Filtered: {len(filtered_df)} rows")
    print(filtered_df.columns)
    print(filtered_df.head())
    
    # Visualize class distribution
    filtered_counts = filtered_df["labels"].value_counts()
    plt.figure(figsize=figsize)
    sns.barplot(y=filtered_counts.index, x=filtered_counts.values, hue=filtered_counts.index, 
                palette="inferno", legend=False)
    plt.title('Amount of images per label', fontsize=12)
    plt.ylabel('Labels', fontsize=10)
    plt.xlabel('Count', fontsize=10)
    
    # Add value labels on bars for clarity
    for i, v in enumerate(filtered_counts.values):
        plt.text(v + 0.5, i, str(v), va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(f'../output/{save_path}', dpi=150)
    plt.show()

In [ ]:
sports_list = ['football', 'formula 1 racing', 'hockey', 'nascar racing', 'rugby', 'canoe slamon', 'basketball']
visualize_sports_distribution(df, sports_list, save_path="seven_sports.png")